In [2]:
# There are N Koinonians and they each have to fill 2 hours of house service every week.
# The house service can be done in shifts of 1 hour or 2 hours.
# Each shift also takes 2 people.
# They are asked which shifts they can work and then a preference for that shift (on some scale, say 1-10).
# The goal is to assign shifts to Koinonians such that:
# 1) Each Koinonian gets exactly 2 hours of service assigned.
# 2) The total preference score across all Koinonians is maximized.
# 3) Every shift has exactly 2 Koinonians assigned to it.
# Design an algorithm to solve this problem and implement it in Python.

In [ ]:
import pandas as pd
import numpy as np
import math
# for pretty loading
import itertools
import threading
import time
import sys
from schedule import readIn, normalizePrefs

pref_file = 'data/test_pref.csv'
shift_file = 'data/shift_data.csv'

pref_data, member_names = readIn(pref_file, shift_file)
# normalizePrefs(pref_data)

In [ ]:
# Load data about each shift from CSV file
df = pd.read_csv(shift_file)
# Split shifts with multiple workers
BIG PROBLEM!
NEED TO ALSO ADD LINES TO THE PREFERENCE DATA!
If a member has availability for a shift with multiple workers
    they must be shown as not available for 2nd, 3rd, etc
    place of that shift!
shift_data
shift_data.head()

SyntaxError: invalid syntax (3811699447.py, line 4)

In [ ]:
# Now, we have to create a list of all possible assignments of a single Koinonian to shifts
# This means, we go through and create all possible ways to assign him 2 hours of shifts.
# We need to store which shifts are assigned in each case, and the total preference score for that assignment,
# which we calculate by adding the preference scores/10 (to get a fraction) for the assigned shifts.

# Start by creating the first person's possible assignments, then with each of those, go through and find the next person's possible assignments, and so on.
# Kind of like DFS and is a tree structure.

In [ ]:
# First, let's store all possible permutations in a table, so we just have to iterate though indexes of the table, and grab the data as needed.
# _____________________________
# | Index | Shift 1 | Shift 2 |
# |   0   |    A    |    B    |
# |   1   |    A    |    C    |
# -----------------------------
# (the index is just for reference, we don't need to store it)


# # We create a shift by choosing a double shift or two single shifts and combining them into one option
# double_shifts = shift_data[shift_data['Duration'] == 2]['Shift'].index
# single_shifts = shift_data[shift_data['Duration'] == 1]['Shift'].index
# n = len(single_shifts)
# n_Cr_2 = math.factorial(n) // (math.factorial(2) * math.factorial(n - 2)) # n choose 2
# possible_assignments = np.empty((n_Cr_2 + len(double_shifts),2), dtype=object) # possible assignmets for a single Koinonian
# # Create doubled single shifts
# iter = 0
# for i in range(len(single_shifts)):
#     for j in range(i+1, len(single_shifts)):
#         possible_assignments[iter,:] = np.array([(single_shifts[i], single_shifts[j])])
#         iter += 1

# for i in range(len(double_shifts)):
#     possible_assignments[iter,:] = np.array([(double_shifts[i], -1)])
#     iter += 1

# print(possible_assignments.shape)
# ### THIS IS WRONGGGG ###
# # It should have every combination of shifts, yes, but then a separate list of what lists are available

In [7]:
# Creating shift arrays:
len_single_shifts = shift_data[shift_data['duration'] == 1].shape[0]
single_shifts = np.ones(len_single_shifts, dtype=int) * 2

len_double_shifts = shift_data[shift_data['duration'] == 2].shape[0]
double_shifts = np.ones(len_double_shifts, dtype=int)

# once we decide on a shift, we have to refer to the key to get the absolute shift index
# this variable stores the absolute index given the relative index in single_shifts and double_shifts
shift_key = (shift_data[shift_data['duration'] == 1].index,
             shift_data[shift_data['duration'] == 2].index)

In [8]:
def pref_score(pref_data, curr_koinonian, assignment):
    if assignment[1] == -1:
        return pref_score_double(pref_data, curr_koinonian, assignment)
    return pref_score_single(pref_data, curr_koinonian, assignment)

def pref_score_single(pref_data, curr_koinonian, assignment):
    return pref_data[curr_koinonian, assignment[0]] + pref_data[curr_koinonian, assignment[1]]

def pref_score_double(pref_data, curr_koinonian, assignment):
    return 2 * pref_data[curr_koinonian, assignment[0]]

In [9]:
def max_score(pref_data, curr_k, single_shifts, double_shifts, shift_key):
    '''
    Finds maximium preference score and corresponding assignment.
    Recursive, but splits into two cases:
    1) Next Koinonian gets two single shifts
    2) Next Koinonian gets a double shift
    and takes the better (in terms of the peoples' overall satisfaction) of the two options.
    '''

    # Base Case: no more Koinonians to assign
    if curr_k >= pref_data.shape[0]:
        return None, 0
    
    # assuming the next person gets a set of single shifts
    best_assignment_single, best_pref_single, valid_single = max_score_single_shifts(pref_data, curr_k, single_shifts, double_shifts, shift_key)

    # assuming the next person gets a double shift
    best_assignment_double, best_pref_double, valid_double = max_score_double_shifts(pref_data, curr_k, single_shifts, double_shifts, shift_key)

    # check which option is valid
    if valid_single:
        if valid_double:
            # both valid, choose best
            if best_pref_single >= best_pref_double:
                return best_assignment_single, best_pref_single
            else:
                return best_assignment_double, best_pref_double
        else:
            # only single valid
            return best_assignment_single, best_pref_single
    # else, only double is valid or neither are
    return best_assignment_double, best_pref_double


def max_score_single_shifts(pref_data, curr_k, single_shifts, double_shifts, shift_key):
    '''
    Inputs:
    - pref_data: np array of shape (no. koin boys, no. shifts), where each element
    where each element represents the preference score of that Koinonian for that shift
    - curr_k: current Koinonian index
    - single_shifts: np array of shape (no. single shifts,), each element represents available slots for that shift
    - double_shifts: np array of shape (no. double shifts,), each element represents available slots for that shift

    Returns:
    - best_assignment: shift assigments most satisfactory for the house overall as an np array of shape (no. koin boys, 2)
        First elt is first shift assigned to Koinonian, second is second shift or -1 if double shift in first place
    - best_pref: highest preference score.
    - boolean indicating whether assignment is valid (i.e., enough shifts available).
    '''

    # Base Case: no more Koinonians to assign
    if curr_k >= pref_data.shape[0]:
        return None, 0, True

    # Initialize best assignment and score
    best_assignment = np.ones((pref_data.shape[0] - curr_k, 2), dtype=int) * -1

    # Case where we don't have enough shifts
    if len(single_shifts[single_shifts > 0]) < 2:
        return best_assignment, 0, False
    
    # Initialize best score
    best_pref = -np.inf

    for i in range(len(single_shifts)): # iterate through all first shifts
        if single_shifts[i] <= 0:
            continue # skip if shift full
        for j in range(i+1, len(single_shifts)): # iterate through all remaining second shifts
            if single_shifts[j] <= 0:
                continue # skip if shift full
            curr_shifts = np.array([shift_key[0][i], shift_key[0][j]])
            # get score of current assignment
            curr_pref = pref_score(pref_data, curr_k, curr_shifts)
            # get cost of remaining Koinonians (recursive to master func)
            next_single_shifts = np.copy(single_shifts)
            next_single_shifts[i] -= 1
            next_single_shifts[j] -= 1
            remaining_assignment, remaining_pref = max_score(pref_data, curr_k + 1, next_single_shifts, double_shifts, shift_key)
            # check if option is more preferred
            if (curr_pref + remaining_pref) > best_pref:
                best_pref = curr_pref + remaining_pref
                # store assignment and cost
                best_assignment[0,:] = curr_shifts
                if remaining_assignment is not None:
                    best_assignment[1:,:] = remaining_assignment

    return best_assignment, best_pref, True


def max_score_double_shifts(pref_data, curr_k, single_shifts, double_shifts, shift_key):
    
    # Base Case: no more Koinonians to assign
    if curr_k >= pref_data.shape[0]:
        return None, 0, True
    
    # Initialize best assignment and score
    best_assignment = np.ones((pref_data.shape[0] - curr_k, 2), dtype=int) * -1

    # Case where we don't have enough shifts
    if len(double_shifts[double_shifts > 0]) < 1:
        return best_assignment, 0, False
    
    # Initialize best score
    best_pref = -np.inf

    for i in range(len(double_shifts)): # iterate through all double shifts
        if double_shifts[i] <= 0:
            continue # skip if shift full
        curr_shifts = np.array([shift_key[1][i], -1])
        # get score of current assignment
        curr_pref = pref_score(pref_data, curr_k, curr_shifts)
        # get cost of remaining Koinonians (recursive to master func)
        next_double_shifts = np.copy(double_shifts)
        next_double_shifts[i] -= 1
        remaining_assignment, remaining_pref = max_score(pref_data, curr_k + 1, single_shifts, next_double_shifts, shift_key)
        # check if option is more preferred
        if (curr_pref + remaining_pref) > best_pref:
            best_pref = curr_pref + remaining_pref
            # store assignment and cost
            best_assignment[0,:] = curr_shifts
            if remaining_assignment is not None:
                best_assignment[1:,:] = remaining_assignment
    
    return best_assignment, best_pref, True

In [ ]:
# def get_preference_score(pref_data, assignment, koinonian):
#     if assignment[1] == -1: # double shifts get their score counted twice
#         return 2*pref_data[koinonian, assignment[0]]
#     return pref_data[koinonian, assignment[0]] + pref_data[koinonian, assignment[1]]

# # Can be made more efficient by skipping unavailable assignments
# def minimum_cost(pref_data, curr_koinonian, available_assignments, possible_assignments):
#     # base case: no more Koinonians to assign
#     if curr_koinonian >= pref_data.shape[0]:
#         return None, 0
    
#     # clean available assignments to only those that have remaining slots
#     available_assignments = available_assignments[available_assignments[:,0] > 0]
    
#     best_assignment = None  # initialize best assignment

#     # recursive case: try all possible assignments for current Koinonian
#     # if there are no available assignment types left, fill remaining with placeholders
#     if len(available_assignments) == 0:
#         remaining_count = pref_data.shape[0] - curr_koinonian
#         if remaining_count > 0:
#             # create an array of [-1, -1] placeholders for each remaining koinonian
#             placeholder = np.array([[-1, -1]], dtype=possible_assignments.dtype)
#             best_assignment = np.tile(placeholder, (remaining_count, 1))
#         else:
#             best_assignment = np.empty((0,2), dtype=possible_assignments.dtype)
#         return best_assignment, 0

#     best_score = get_preference_score(pref_data, possible_assignments[available_assignments[0, 1], :], curr_koinonian) - 1  # initialize to a low value

#     # try each available assignment option
#     for i in range(len(available_assignments)):
#         curr_assignment = possible_assignments[available_assignments[i, 1], :]
        
#         # get preference score for this assignment
#         curr_score = get_preference_score(pref_data, curr_assignment, curr_koinonian)
#         if curr_score < 0:
#             continue # skip unavailable assignments

#         # get minimum preference score for remaining Koinonians recursively
#         new_available_assignments = np.copy(available_assignments)
#         new_available_assignments[i,0] -= 1
#         best_remaining_assignment, min_remaining_score = minimum_cost(pref_data, curr_koinonian + 1, new_available_assignments, possible_assignments)

#         total_score = curr_score + min_remaining_score
#         # iterate through all possible assignments and find the maximum total score
#         if total_score > best_score:
#             best_score = total_score
#             if best_remaining_assignment is None:
#                 best_assignment = np.array([curr_assignment])
#             else:
#                 best_assignment = np.vstack((np.array([curr_assignment]), best_remaining_assignment))
    
#     # If nothing was selectable at this level (all were unavailable), return placeholders for remaining koinonians
#     if best_assignment is None:
#         remaining_count = pref_data.shape[0] - curr_koinonian
#         if remaining_count > 0:
#             placeholder = np.array([[-1, -1]], dtype=possible_assignments.dtype)
#             best_assignment = np.tile(placeholder, (remaining_count, 1))
#         else:
#             best_assignment = np.empty((0,2), dtype=possible_assignments.dtype)
#         return best_assignment, 0

#     # return best score and assignment
#     return best_assignment, best_score

In [ ]:
curr_koinonian = 0
# available_assignments = np.array([np.ones(len(possible_assignments),dtype=int)*2, range(len(possible_assignments))]).T # first column is the number of these shifts remaining, second column is the shift index for possible_assigments

# print(available_assignments.shape)

done = False
def animate():
    for c in itertools.cycle(['|', '/', '-', '\\']):
        if done:
            break
        sys.stdout.write('\rCalculating ' + c)
        sys.stdout.flush()
        time.sleep(0.1)
    sys.stdout.write('\rDone!     \n')
t = threading.Thread(target=animate)
t.start()

# best_assignment, best_score = minimum_cost(pref_data, curr_koinonian, available_assignments, possible_assignments)
best_assignment, best_score = max_score(pref_data, curr_koinonian, single_shifts, double_shifts, shift_key)
done = True
t.join()

print('Best Assignment:\n', best_assignment, '\n')
print('Best Score:\n', best_score)
print(best_assignment.shape)

Calculating |

IndexError: index 4 is out of bounds for axis 1 with size 4

Calculating -

In [11]:
done = True

Done!     


In [ ]:
# Calculate absolute overall scores:
absolute_scores = np.zeros(abs_pref_data.shape[0])
for i in range(abs_pref_data.shape[0]):
    assigned_shift = best_assignment[i]
    absolute_scores[i] = get_preference_score(abs_pref_data, assigned_shift, i)
print('Absolute Scores:\n', absolute_scores, '\n')
print('Absolute Total Score:\n', np.sum(absolute_scores))

# Calculate relative overall scores:
relative_scores = np.zeros(pref_data.shape[0])
for i in range(pref_data.shape[0]):
    assigned_shift = best_assignment[i]
    relative_scores[i] = get_preference_score(pref_data, assigned_shift, i)

print('Relative Scores:\n', relative_scores, '\n')
print('Relative Total Score:\n', np.sum(relative_scores / (2*len(relative_scores))))

In [3]:
import torch
import torch.optim as optim

In [ ]:
def koinSmoothFunc():
    pass

def MSELoss():
    pass

In [ ]:

# Define target function
f = koinSmoothFunc()# give inputs

# Define the loss function (MSELoss) and Adam optimizer
criterion = MSELoss()
optimizer = optim.Adam(f.parameters(), lr=0.01)

# Sample training loop
for epoch in range(50):
  # Sample input and target
  x = torch.randn(5, 1)

  # Forward pass
  output = f(x)
  # loss = criterion(output, y)

  # Backward pass and optimization
  optimizer.zero_grad()
  # loss.backward()
  optimizer.step()

  # Print loss every 10 epochs
  if epoch % 10 == 0:
    print(f'Epoch [{epoch}/100]')
          # , Loss: {loss.item():.4f}')


Epoch [0/100], Loss: 14.4264
Epoch [10/100], Loss: 16.0855
Epoch [20/100], Loss: 9.6520
Epoch [30/100], Loss: 22.1855
Epoch [40/100], Loss: 6.8812
